# Technique 3 — Structure with XML Tags

When a prompt interpolates a lot of content, Claude can struggle to tell where *your data* ends and *your instructions* begin. **XML tags** create clear boundaries.

- Use **descriptive, custom** tag names — `<sales_records>` beats `<data>`; `<my_code>` + `<docs>` separate content types; `<athlete_information>` labels the user's details.
- Most valuable when: including large context/data, mixing content types (code + docs + data), or interpolating multiple variables.

For the meal planner we wrap the athlete inputs in `<athlete_information>`:

```
<athlete_information>
- Height: ...
- Weight: ...
- Goal: ...
- Dietary restrictions: ...
</athlete_information>
```

> Incremental on top of being-specific (guidelines stay). **The course flags this as a modest gain for simple prompts** — and on Haiku 4.5, already near the ceiling, expect it roughly flat. XML structure earns its keep as prompts grow large and mixed, not on a four-field input. Same harness, same `dataset.json`.

## Load the harness + existing dataset

In [1]:
import sys
import os
from dotenv import find_dotenv

REPO_ROOT = os.path.dirname(find_dotenv())
SECTION = os.path.join(REPO_ROOT, "building-with-the-claude-api", "03-prompt-engineering")
if SECTION not in sys.path:
    sys.path.insert(0, SECTION)

from prompt_evaluator import PromptEvaluator, add_user_message, chat

DATASET = os.path.join(SECTION, "dataset.json")
evaluator = PromptEvaluator(max_concurrent_tasks=3)
print("Harness loaded. Reusing dataset ->", DATASET)

Harness loaded. Reusing dataset -> c:\Development\anthropic-cert\building-with-the-claude-api\03-prompt-engineering\dataset.json


## Improved prompt — wrap inputs in `<athlete_information>`

Guidelines unchanged; the only difference is the athlete's four fields are now clearly delimited as a single related block.

In [2]:
def run_prompt(prompt_inputs):
    prompt = f"""
Generate a one-day meal plan for an athlete that meets their dietary restrictions.

<athlete_information>
- Height: {prompt_inputs["height"]}
- Weight: {prompt_inputs["weight"]}
- Goal: {prompt_inputs["goal"]}
- Dietary restrictions: {prompt_inputs["restrictions"]}
</athlete_information>

Guidelines:
1. Include accurate daily calorie amount
2. Show protein, fat, and carb amounts
3. Specify when to eat each meal
4. Use only foods that fit restrictions
5. List all portion sizes in grams
6. Keep budget-friendly if mentioned
"""

    messages = []
    add_user_message(messages, prompt)
    return chat(messages)

## Re-evaluate

Compare against being-specific. Don't be surprised if it's flat or wiggles by noise — a four-field block is exactly the "simple prompt" case where the course says XML structure won't move the needle much.

In [3]:
results = evaluator.run_evaluation(
    run_prompt_function=run_prompt,
    dataset_file=DATASET,
    extra_criteria="""
The output should include:
- Daily caloric total
- Macronutrient breakdown
- Meals with exact foods, portions, and timing
""",
    json_output_file=os.path.join(SECTION, "output-04-xml-tags.json"),
    html_output_file=os.path.join(SECTION, "output-04-xml-tags.html"),
)

Graded 1/3 test cases
Graded 2/3 test cases
Graded 3/3 test cases
Average score: 7.666666666666667


## Takeaway

XML tags are a **scalability** technique: the payoff grows with prompt size and content variety, not with a tiny input. Even when the score doesn't move on a simple prompt, adopting the habit keeps big prompts unambiguous — and it sets up the next technique cleanly, since **providing examples** adds its own `<sample_input>` / `<ideal_output>` blocks that need to be visually distinct from the real inputs.